# Import libraries

In [1]:
!pip install split-folders
!pip install tqdm
!pip install scikit-image
!pip install opencv-python
!pip install scikit-learn
!pip install tensorflow

In [2]:
import os, shutil
import splitfolders
import random
import numpy as np
import pandas as pd
import itertools
from tqdm import tqdm, tqdm_notebook
import cv2
from scipy import stats
from sklearn.metrics import confusion_matrix, roc_curve,auc, classification_report, precision_score, recall_score
from sklearn.linear_model import LinearRegression

import skimage
import skimage.segmentation
import copy

# Data Visualization
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('ggplot')
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.image as mpimg


import tensorflow as tf
print(tf.__version__)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Input, Flatten, Dense, MaxPooling2D, Conv2D, Dropout
from tensorflow.keras.applications.vgg19 import VGG19
from tensorflow.keras.optimizers import SGD, RMSprop, Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications.imagenet_utils import decode_predictions

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
from tensorflow.keras.layers import GlobalAveragePooling2D, GlobalMaxPooling2D
from tensorflow.keras.layers import Dense, Reshape, Multiply, Add, Conv2D, Activation, Concatenate

def cbam_block(feature_map, ratio=8):
    # Channel Attention
    avg_pool = GlobalAveragePooling2D()(feature_map)
    max_pool = GlobalMaxPooling2D()(feature_map)

    shared_dense1 = Dense(feature_map.shape[-1] // ratio, activation='relu')
    shared_dense2 = Dense(feature_map.shape[-1], activation='sigmoid')

    avg_out = shared_dense2(shared_dense1(avg_pool))
    max_out = shared_dense2(shared_dense1(max_pool))

    channel_attention = Add()([avg_out, max_out])
    channel_attention = Reshape((1,1,feature_map.shape[-1]))(channel_attention)

    channel_refined = Multiply()([feature_map, channel_attention])

    # Spatial Attention
    avg_spatial = tf.keras.layers.Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(channel_refined)
    max_spatial = tf.keras.layers.Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(channel_refined)


    spatial = Concatenate(axis=-1)([avg_spatial, max_spatial])
    spatial_attention = Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(spatial)

    refined_feature = Multiply()([channel_refined, spatial_attention])

    return refined_feature


# Load data and split the data into training, validation and testing datasets

In [ ]:
original_dataset = r'C:\Users\HP\Downloads\malaria-detection-combined-main\cell_images'
original_dataset_parasitized = os.path.join(original_dataset, 'Parasitized')
original_dataset_uninfected = os.path.join(original_dataset, 'Uninfected')

Copying images into training, validation and test directories

In [ ]:
# Create a base dir
if not os.path.isdir('healthy_and_infected'):
                base_dir = r'C:\Users\HP\Downloads\malaria-detection-combined-main\healthy_and_infected'
                os.makedirs(base_dir,exist_ok=True)

In [ ]:
# Make train, valid and test directories
#train
if not os.path.isdir('healthy_and_infected/train'):
    train_dir = os.path.join(base_dir, 'train')
    os.makedirs(train_dir,exist_ok=True)
#valid
if not os.path.isdir('healthy_and_infected/valid'):
    valid_dir = os.path.join(base_dir, 'valid')
    os.makedirs(valid_dir,exist_ok=True)
#test
if not os.path.isdir('healthy_and_infected/test'):
    test_dir = os.path.join(base_dir, 'test')
    os.makedirs(test_dir,exist_ok=True)

In [ ]:
# Make directories for infected images in each of the train, valid and test directories
if not os.path.isdir('healthy_and_infected/train/infected'):
    infected_trn_dir = os.path.join(train_dir, 'infected')
    os.makedirs(infected_trn_dir,exist_ok=True)

if not os.path.isdir('healthy_and_infected/valid/infected'):
    infected_valid_dir = os.path.join(valid_dir, 'infected')
    os.makedirs(infected_valid_dir,exist_ok=True)

if not os.path.isdir('healthy_and_infected/test/infected'):
    infected_test_dir = os.path.join(test_dir, 'infected')
    os.makedirs(infected_test_dir,exist_ok=True)

In [ ]:
# Make directories for healthy images in each of the train, valid and test directories
if not os.path.isdir('healthy_and_infected/train/healthy'):
    healthy_trn_dir = os.path.join(train_dir, 'healthy')
    os.makedirs(healthy_trn_dir,exist_ok=True)

if not os.path.isdir('healthy_and_infected/valid/healthy'):
    healthy_valid_dir = os.path.join(valid_dir, 'healthy')
    os.makedirs(healthy_valid_dir,exist_ok=True)

if not os.path.isdir('healthy_and_infected/test/healthy'):
    healthy_test_dir = os.path.join(test_dir, 'healthy')
    os.makedirs(healthy_test_dir,exist_ok=True)

In [ ]:
# Copies the first 10981 infected images to the infected_train_dir
fnames = [f for f in os.listdir(original_dataset_parasitized) if f.endswith('.png')][:10980]
for fname in fnames:
    src = os.path.join(original_dataset_parasitized, fname)
    dst = os.path.join(infected_trn_dir, fname)
    shutil.copyfile(src,dst)

In [ ]:
# Copies the 1400 infected images (10981, 12382) to the infected_valid_dir
fnames = [f for f in os.listdir(original_dataset_parasitized) if f.endswith('.png')][10981:12382]
for fname in fnames:
    src = os.path.join(original_dataset_parasitized, fname)
    dst = os.path.join(infected_valid_dir, fname)
    shutil.copyfile(src,dst)

In [ ]:
### Copy another 1400 infected images () to the infected_test_dir
fnames = [f for f in os.listdir(original_dataset_parasitized) if f.endswith('.png')][12382:13780]
for fname in fnames:
    src = os.path.join(original_dataset_parasitized, fname)
    dst = os.path.join(infected_test_dir, fname)
    shutil.copyfile(src,dst)

In [ ]:
# Copies the first 10981 uninfected images to the healthy_train_dir
fnames = [f for f in os.listdir(original_dataset_uninfected) if f.endswith('.png')][:10981]
for fname in fnames:
    src = os.path.join(original_dataset_uninfected, fname)
    dst = os.path.join(healthy_trn_dir, fname)
    shutil.copyfile(src,dst)

In [ ]:
# Copies the 1400 uninfected images (10981, 12382) to the healthy_valid_dir
fnames = [f for f in os.listdir(original_dataset_uninfected) if f.endswith('.png')][10981:12382]
for fname in fnames:
    src = os.path.join(original_dataset_uninfected, fname)
    dst = os.path.join(healthy_valid_dir, fname)
    shutil.copyfile(src,dst)

In [ ]:
### Copy another 1400 uninfected images () to the healthy_test_dir
fnames = [f for f in os.listdir(original_dataset_uninfected) if f.endswith('.png')][12382:13780]
for fname in fnames:
    src = os.path.join(original_dataset_uninfected, fname)
    dst = os.path.join(healthy_test_dir, fname)
    shutil.copyfile(src,dst)

In [ ]:
print("{} Infected training images".format(len(os.listdir(infected_trn_dir))))
print("{} Uninfected training images".format(len(os.listdir(healthy_trn_dir))))
print(" {} Infected valid images".format(len(os.listdir(infected_valid_dir))))
print(" {} Uninfected valid images".format(len(os.listdir(healthy_valid_dir))))
print(" {} Infected testing images".format(len(os.listdir(infected_test_dir))))
print(" {} Uninfected testing images".format(len(os.listdir(healthy_test_dir))))

# Exploratory Data Analysis

In [ ]:
# Train
infected_trn_fpaths = [os.path.join(infected_trn_dir, fpath) for fpath in os.listdir(infected_trn_dir)]
healthy_trn_fpaths = [os.path.join(healthy_trn_dir, fpath) for fpath in os.listdir(healthy_trn_dir)]

# Valid
infected_valid_fpaths = [os.path.join(infected_valid_dir, fpath) for fpath in os.listdir(infected_valid_dir)]
healthy_valid_fpaths = [os.path.join(healthy_valid_dir, fpath) for fpath in os.listdir(healthy_valid_dir)]

# Test
infected_test_fpaths = [os.path.join(infected_test_dir, fpath) for fpath in os.listdir(infected_test_dir)]
healthy_test_fpaths = [os.path.join(healthy_test_dir, fpath) for fpath in os.listdir(healthy_test_dir)]


In [ ]:
def get_img_shape(idx, img, total_num_images):
    
    if idx%2000 ==0 or idx == (total_num_images-1):
        print("working on img {}".format(idx))
    return cv2.imread(img).shape

data_inp = [(idx, img, len(infected_trn_fpaths + healthy_trn_fpaths)) for idx, img in enumerate(infected_trn_fpaths + healthy_trn_fpaths)]

train_img_dims_map = list(map(get_img_shape, [input[0] for input in data_inp],
    [input[1] for input in data_inp],
    [input[2] for input in data_inp]))

In [ ]:
print('Min Dimensions:           {}'.format(np.min(train_img_dims_map, axis=0)))
print('Avg Dimensions:           {}'.format(np.mean(train_img_dims_map, axis=0)))
print('Median Dimensions:        {}'.format(np.median(train_img_dims_map, axis=0)))
print('Most Frequent Dimensions: {}'.format(stats.mode(train_img_dims_map, axis=0)[0]))
print('Max Dimensions:           {}'.format(np.max(train_img_dims_map, axis=0)))

In [ ]:
infected_trn_samples = random.sample(infected_trn_fpaths, 5)
healthy_trn_samples = random.sample(healthy_trn_fpaths, 5)

In [ ]:
fig =plt.figure(figsize=(28,14))
columns=5
rows=1
for i in range(1, columns*rows +1):
    fig.add_subplot(rows, columns, i)
    plt.imshow(mpimg.imread(infected_trn_samples[i-1]))
    plt.axis('off')
    plt.title('Infected', fontsize=32)
plt.show()


fig =plt.figure(figsize=(28,14))
columns=5
rows=1
for i in range(1, columns*rows +1):
    fig.add_subplot(rows, columns, i)
    plt.imshow(mpimg.imread(healthy_trn_samples[i-1]))
    plt.axis('off')
    plt.title('Healthy', fontsize=32)
plt.show()


# Data Augumentation and resizing images

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255.,
                                   horizontal_flip=True,
                                   vertical_flip=0.4,
                                   rotation_range=30,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   brightness_range=[0.8,1.2],
                                   fill_mode='nearest')
valid_datagen = ImageDataGenerator(rescale=1.0/255.)
test_datagen = ImageDataGenerator(rescale=1.0/255.)

train_generator = train_datagen.flow_from_directory(train_dir,
                                                    batch_size=32,
                                                    target_size=(128,128),
                                                    class_mode='categorical',
                                                    shuffle=True,
                                                    seed=42,
                                                    color_mode='rgb')

valid_generator = valid_datagen.flow_from_directory(valid_dir,
                                                    batch_size=32,
                                                    target_size=(128, 128),
                                                    class_mode='categorical',
                                                    shuffle=True,
                                                    seed=42,
                                                    color_mode='rgb')

class_labels = train_generator.class_indices
class_names = {value:key for (key, value) in class_labels.items()}

In [ ]:
class_labels, class_names

# Unfreezing and Fine-tuning the entire network

In [ ]:
# Instantiate VGG19 model with weights from Imagenet without the calssifier at the top
base_model = VGG19(input_shape = (128,128,3),
                   include_top = False, 
                   weights = 'imagenet')
# Freeze the ConvNet to avoid weight updates
for layer in base_model.layers:
    layer.trainable=False
    
x = base_model.output
flat=Flatten()(x)

# Add a classifier -  a fully connected dense layers
class_1 = Dense(4608, activation='relu')(flat)
drop_out = Dropout(0.2)(class_1)
class_2 = Dense(1152, activation='relu')(drop_out)
output = Dense(2, activation='softmax')(class_2)

# Call backs
filepath = 'best_model.h5'
es = EarlyStopping(monitor='val_loss', verbose=1, mode='min', patience=4)
cp = ModelCheckpoint(filepath, monitor='val_loss', verbose=1, save_best_only=True,
                     save_weights_only=False, mode='auto', save_freq='epoch')
lrr = ReduceLROnPlateau(monitor='val_accuracy',patience=3,verbose=1,factor=0.5,min_lr=0.0001)

# Define an optimizer
sgd = SGD(learning_rate=.0001, decay=1e-6, momentum=0.9, nesterov=True)



In [ ]:
# Build the network
base_model = VGG19(include_top=False, input_shape=(128,128,3))
x = base_model.output
x = cbam_block(x)
flat=Flatten()(x)
class_1 = Dense(4608, activation='relu')(flat)
drop_out = Dropout(0.2)(class_1)
class_2 = Dense(1152, activation='relu')(drop_out)
output = Dense(2, activation='softmax')(class_2)
model_03 = Model(base_model.inputs, output)

sgd = SGD(learning_rate=0.0001, momentum=0.9, nesterov=True)
# Compile the model
model_03.compile(optimizer=sgd, loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
history_03 = model_03.fit(train_generator,
                                    steps_per_epoch=100,
                                    epochs=10,
                                    callbacks = [es, cp, lrr],
                                    validation_data = valid_generator)

In [ ]:
# save model
if not os.path.isdir('model_weights/'):
    os.mkdir('model_weights/')
model_03.save_weights(filepath='model_weights/vgg_unfrozen.weights.h5', overwrite=True)

## Evaluating the model

In [ ]:
# Resize test images similar to the train data
test_generator = test_datagen.flow_from_directory(test_dir,
                                                  batch_size=1,
                                                  target_size=(128, 128),
                                                  class_mode='categorical',
                                                  shuffle=False,
                                                  seed=42,
                                                  color_mode='rgb')

In [ ]:
# Load the saved model
model_03.load_weights('model_weights/vgg_unfrozen.weights.h5')

# Evaluate the model on the hold out validation and test datasets

vgg_val_eval_03 = model_03.evaluate(valid_generator)
vgg_test_eval_03 = model_03.evaluate(test_generator)

print('Validation loss     :{0:.2f}'.format(vgg_val_eval_03[0]))
print('Validation accuracy :{0:.2f}'.format(vgg_val_eval_03[1]))
print('Test loss           :{0:.2f}'.format(vgg_test_eval_03[0]))
print('Test accuracy       :{0:.2f}'.format(vgg_test_eval_03[1]))


filenames = test_generator.filenames
nb_samples = len(filenames)
vgg_predictions_03 = model_03.predict(test_generator,
                                                steps = nb_samples,
                                                verbose=1)
vgg_pred_labels_03 = np.argmax(vgg_predictions_03, axis=1)


# Classification Report
print(classification_report(test_generator.classes, vgg_pred_labels_03, 
                            target_names=['healthy', 'infected']))
vgg_conf_mat_03 = pd.DataFrame(confusion_matrix(test_generator.classes, vgg_pred_labels_03), 
                        index=['healthy', 'infected'], 
                        columns=['healthy', 'infected'])


fig, ax = plt.subplots(figsize=(5,5))

sns.heatmap(vgg_conf_mat_03, annot=True, fmt=".1f", linewidths=0.5, square=True, cmap='Blues_r')
ax.set_ylabel("Actual Label", fontsize=14)
ax.set_xlabel("Predicted Label", fontsize=14)
all_sample_title="Accuracy Score: {0:.2f}".format(vgg_test_eval_03[1])
ax.set_title(all_sample_title, size=15)
ax.set_ylim(len(vgg_conf_mat_03)-0.05, -0.05)
plt.tight_layout()

In [ ]:
# AUC Curve
false_positive_rate, true_positive_rate, threshold = roc_curve(test_generator.classes, vgg_pred_labels_03)
area_under_curve = auc(false_positive_rate, true_positive_rate)

# Plot AUC Curve
fig, ax = plt.subplots(figsize=(12,8))
ax.plot([0,1], [0,1], 'k--')
ax.plot(false_positive_rate, true_positive_rate, label='AUC = {:.3f}'.format(area_under_curve))
ax.set_xlabel('False Positive Rate', fontsize=16)
ax.set_ylabel('True Positive Rate', fontsize=16)
ax.set_title("ROC curve", fontsize=20)
ax.legend(frameon=False, loc='best', ncol=1, fontsize=16)

# Prediction Results for 10 randomly selected images

In [ ]:
test_images = [img for img in random.sample(infected_test_fpaths, 5)]
test_images.extend([img for img in random.sample(healthy_test_fpaths, 5)])


In [ ]:
from tensorflow.keras.preprocessing import image
true_labels = []
predicted_labels = [] 
fig = plt.figure(figsize=(28,14))
columns=5
rows=2
for i in range(1, columns*rows +1):
    fig.add_subplot(rows, columns, i)
    true_label = os.path.basename(os.path.dirname(test_images[i-1]))
    img = mpimg.imread(test_images[i-1])
    plt.imshow(img)
    plt.axis('off')
    img_for_model = image.load_img(test_images[i-1], target_size=(128,128))
    img_for_model = image.img_to_array(img_for_model)
    img_for_model = np.expand_dims(img_for_model, axis=0)
    prediction = model_03.predict(img_for_model)
    predicted_label = np.argmax(prediction)
    plt.title(f"Predicted: {class_names[predicted_label]}\nActual: {true_label}", fontsize=16)
plt.tight_layout()
plt.show()

# Interpretable the classifier with LIME for Image Classification

In [ ]:
!pip install lime

In [ ]:
%load_ext autoreload
%autoreload 2
import os,sys
import lime
from lime import lime_image
from skimage.segmentation import mark_boundaries

In [ ]:
test_image = skimage.io.imread(test_images[0])
test_image = skimage.transform.resize(test_image, (128,128))
import matplotlib.pyplot as plt
plt.imshow(test_image)
plt.grid(True)       
plt.axis('on')       
plt.show()


# img = image.load_img(test_images[i-1], target_size=(128,128))
# img = image.img_to_array(img)
# img = np.expand_dims(img, axis = 0)
# prediction = model_03.predict(img)
# prediction

In [ ]:
explainer = lime_image.LimeImageExplainer()

In [ ]:
%%time
# Hide color is the color for a superpixel turned OFF. Alternatively, if it is NONE, the superpixel will be replaced by the average of its pixels
explanation = explainer.explain_instance(test_image, model_03.predict, top_labels=1, hide_color=0)

In [ ]:
temp, mask = explanation.get_image_and_mask(explanation.top_labels[0], positive_only=True, num_features=30, hide_rest=False)
plt.imshow(mark_boundaries(temp / 2 + 0.5, mask))

**Refrences:**
    1. Deep Learning with Python: François Chollet
    
    2. Malaria Detection - Deep Learning Healthcare Case-Study https://github.com/dipanjanS/data_science_for_all/blob/master/os_malaria_detection/Malaria%20Detection%20-%20Deep%20Learning%20Healthcare%20Case-Study.ipynb
    
    3. LIME Tutorial https://github.com/marcotcr/lime/blob/master/doc/notebooks/Tutorial%20-%20Image%20Classification%20Keras.ipynb
     